In [ ]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [ ]:


import chromadb

#Probamos modo efimero

client = chromadb.EphemeralClient()

Las colecciones agrupan vectores que comparten el mismo contexto. Al crearla, Chroma le asigna automáticamente su modelo de embedding por defecto (all-MiniLM-L6-v2)

In [ ]:
# Creamos una colección llamada "articulos_tecnologia"
coleccion = client.get_or_create_collection(name="articulos_tecnologia")

Para meter información en ChromaDB, usamos el método .add(). Lo genial es que le pasás texto plano (strings) y ella sola hace el trabajo pesado: traduce el texto a números (embeddings) y los guarda.

In [ ]:
# La información minima necesaria es el documento (texto) y un ID - La metadata es opcional pero recomendada

coleccion.add(
    documents=[
        "Las bases de datos vectoriales son clave para arquitecturas RAG.",
        "El motor tradicional SQL se basa en coincidencias exactas y tablas.",
        "ChromaDB utiliza el algoritmo HNSW para búsquedas de proximidad rápidas."
    ],
    metadatas=[
        {"categoria": "IA", "autor": "Martin"},
        {"categoria": "Bases de Datos", "autor": "Elena"},
        {"categoria": "IA", "autor": "Martin"}
    ],
    ids=["doc_1", "doc_2", "doc_3"]
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 22.1MiB/s]


Una vez que los datos están indexados, podés hacer consultas a la base de datos usando lenguaje natural con .query().

Chroma convertirá la pregunta en un vector y buscará por aproximación matemático-espacial cuáles son los documentos más cercanos, ordenándolos por su distancia (similitud).

In [ ]:
# Buscamos información relacionada, usando palabras que NO están textualmente en la base
resultados = coleccion.query(
    query_texts=["Hablame sobre herramientas para Vases de datos"],
    n_results=2 # Le pedimos los 2 resultados más cercanos
)

# Mostrar los documentos encontrados
print(resultados["documents"])

[['Las bases de datos vectoriales son clave para arquitecturas RAG.', 'ChromaDB utiliza el algoritmo HNSW para búsquedas de proximidad rápidas.']]


Que sucede si queremos insertar nuevos documentos en una colección? - Lo que equivale a un insert en un sistema relacional





In [ ]:
# 1. Instanciás o recuperás la colección existente
# (Usar 'get_or_create_collection' asegura que si ya existe, simplemente la abra)
coleccion = client.get_or_create_collection(name="articulos_tecnologia")

# 2. Agregás los nuevos documentos
#  Asegurase de usar IDs nuevos que no existan en la base
coleccion.add(
    documents=[
        "Los Large Language Models (LLMs) requieren bases vectoriales para gestionar su memoria a largo plazo.",
        "El filtrado híbrido permite combinar metadatos estructurados con búsquedas semánticas."
    ],
    metadatas=[
        {"categoria": "IA", "autor": "Elena"},
        {"categoria": "Bases de Datos", "autor": "Martin"}
    ],

    # Lo correcto es crear una funcion de generación de indices que valide que no se pisen los indices
    ids=["doc_4", "doc_5"]  # Continuamos la secuencia de IDs anteriores
)

print(f"Documentos agregados. Total actual en la colección: {coleccion.count()}")

Documentos agregados. Total actual en la colección: 5


In [ ]:
# Vamos a realizar nuevamente la busqueda con una colección extendida

# Buscamos información relacionada, usando palabras que NO están textualmente en la base
resultados = coleccion.query(
    query_texts=["Hablame sobre herramientas para Vases de datos"],
    n_results=2 # Le pedimos los 2 resultados más cercanos
)

# Mostrar los documentos encontrados
print(resultados["documents"])

[['Las bases de datos vectoriales son clave para arquitecturas RAG.', 'El filtrado híbrido permite combinar metadatos estructurados con búsquedas semánticas.']]


Vamos a intentar romper la logica violando la unicidad de los indices

In [ ]:
coleccion = client.get_or_create_collection(name="articulos_tecnologia")
coleccion.add(
    documents=[
        "Este texto deberia generar un conflicto de ID" ],
    metadatas=[
                {"categoria": "Bases de Datos", "autor": "Martin"}
    ],

    # Lo correcto es crear una funcion de generación de indices que valide que no se pisen los indices
    ids=["doc_5"]  # Continuamos la secuencia de IDs anteriores
)

print(f"Documentos agregados. Total actual en la colección: {coleccion.count()}")

Documentos agregados. Total actual en la colección: 5


Como podemos ver el add no se concluyo y el documento no se incluyo debido a la inconsistencia del ID.

## Actualizacion

In [ ]:
coleccion.upsert(
    documents=["Texto modificado o actualizado para el documento 1."],
    metadatas=[{"categoria": "IA", "estado": "actualizado"}],
    ids=["doc_1"] # Modifica el doc_1 original
)

In [ ]:
print(f"Documentos agregados. Total actual en la colección: {coleccion.count()}")

Documentos agregados. Total actual en la colección: 5


In [ ]:
# Veremos los cambios generados usando la busqueda hibrida de metadata

# Buscamos información semántica, pero filtrando por metadatos
resultados = coleccion.query(
    query_texts=["Texto"],
    n_results=2,
    # Filtro estricto: solo documentos donde la categoría sea exactamente 'IA'
    where={"categoria": "IA"}
)

print(resultados["documents"])

[['Texto modificado o actualizado para el documento 1.', 'Los Large Language Models (LLMs) requieren bases vectoriales para gestionar su memoria a largo plazo.']]


Podria haber sido una salida mas exacta si refinamos el filtro

In [ ]:
# Aplicamos operadores logicos como AND

# Buscamos información semántica, pero filtrando por metadatos
resultados = coleccion.query(
    query_texts=["Texto"],
    n_results=2,
    # Filtro estricto: solo documentos donde la categoría sea exactamente 'IA'
    where={"$and":
     [
        {"categoria": "IA"} ,{"estado": "actualizado"}
        ]}
)

print(resultados["documents"])

[['Texto modificado o actualizado para el documento 1.']]


Es posible usar operadores de comparación tambien

$eq: Igual a (Equal to) -> {"campo": {"$eq": "valor"}}

$ne: No igual a (Not Equal to) -> {"campo": {"$ne": "valor"}}

$gt: Mayor que (Greater Than) -> {"paginas": {"$gt": 10}} (Para metadatos numéricos)

$gte: Mayor o igual que (Greater Than or Equal)

$lt: Menor que (Less Than)

$lte: Menor o igual que (Less Than or Equal)

Vamos a crear otra colección que nos permita practicar el uso de operadores

In [ ]:
# Creamos una colección llamada "datos_carrera"
coleccion = client.get_or_create_collection(name="datos_carrera")

textos = [
        "El Ultra de tres quebrachos, UT3Q es un ultra muy duro en sus mayores distancias",
        "Las distancias mas cortas de 3 Quebrachos tenian un componente exigente",
        "Las carreras intermedias suelen ser las mas tecnicas.",
        "Las ultimas dos carreras de 100k en argentina tuvieron poca participacion",
        "La distancia mas larga de Punta Negra en San Juan tuvo poca convocatoria",
        "Los corredores estan eligiendo las carreras de elite para participar",
        "En Turmalina hay distancias medias",
        "La Maraton de Córdoba se va a desarrollar el 6 de Julio"
    ]


# Vemos cuantos registros tiene nuestra colección para evitar pisar los indices
cantidad = coleccion.count()
lista_ids =[]

for i in range(len(textos)):
  lista_ids.append(f"doc_{cantidad +i}")


coleccion.add(
    documents= textos,
    metadatas=[
        {"categoria": "Trail", "distancia": 97,"altimetria":3300},
        {"categoria": "Trail", "distancia": 25,"altimetria":1300},
        {"categoria": "Trail", "distancia": 60,"altimetria":3000},
        {"categoria": "Trail", "distancia": 100,"altimetria":3500},
        {"categoria": "Trail", "distancia": 101,"altimetria":3800},
        {"categoria": "Trail", "distancia": 110,"altimetria":6300},
        {"categoria": "Trail", "distancia": 42,"altimetria":2500},
        {"categoria": "Calle", "distancia": 42,"altimetria":400},
    ],
    ids=lista_ids
)

In [ ]:
# Buscamos información semántica, pero filtrando por metadatos
resultados = coleccion.query(
    query_texts=["participantes"],
    n_results=3,

    where={"$and":
     [
        {"categoria": "Trail"} ,{"distancia":{"$gte":40}}
        ]}
)


for resultado in resultados["documents"][0]:
  print("-----")
  print(resultado)

-----
Los corredores estan eligiendo las carreras de elite para participar
-----
Las carreras intermedias suelen ser las mas tecnicas.
-----
Las ultimas dos carreras de 100k en argentina tuvieron poca participacion


In [ ]:
# Buscamos información semántica, pero filtrando por metadatos
resultados = coleccion.query(
    query_texts=["juan"],
    n_results=3,

    where={"$and":
     [
        {"categoria": "Trail"} ,{"distancia":{"$gte":40}}
        ]}
)


for resultado in resultados["documents"][0]:
  print("-----")
  print(resultado)

-----
La distancia mas larga de Punta Negra en San Juan tuvo poca convocatoria
-----
Las carreras intermedias suelen ser las mas tecnicas.
-----
Los corredores estan eligiendo las carreras de elite para participar


In [ ]:
# Buscamos información semántica, pero filtrando por metadatos
resultados = coleccion.query(
    query_texts=["quebracho"],
    n_results=3,

    where={"$and":
     [
        {"categoria": "Trail"} ,{"distancia":{"$gte":20}}
        ]}
)


for resultado in resultados["documents"][0]:
  print("-----")
  print(resultado)

-----
El Ultra de tres quebrachos, UT3Q es un ultra muy duro en sus mayores distancias
-----
Las carreras intermedias suelen ser las mas tecnicas.
-----
La distancia mas larga de Punta Negra en San Juan tuvo poca convocatoria


In [ ]:
# Buscamos información semántica, pero filtrando por metadatos
resultados = coleccion.query(
    query_texts=["quebracho"],
    n_results=3,

    where=

        {"categoria": "Trail"}

)


for resultado in resultados["documents"][0]:
  print("-----")
  print(resultado)

-----
El Ultra de tres quebrachos, UT3Q es un ultra muy duro en sus mayores distancias
-----
Las carreras intermedias suelen ser las mas tecnicas.
-----
La distancia mas larga de Punta Negra en San Juan tuvo poca convocatoria


Cada una de las pruebas anteriores busco darle prioridad a la busqueda semantica de la palabra "quebracho" ya que uno de los id no era retornado, no tuvimos exito pero sirvio para ver como si bien el modelo de busqueda semantica de ChromaDB es una herramienta poderosa, sigue teniendo serias limitaciones.

## ACTUALIZACIONES

In [ ]:
# El metodo UPDATE nos permite actualizar registros que existan, para esto tendremos que especificar el ids, y luego lo que vamos a actualizar, podria ser el o los documents, y el o los metadatos
# En este ejemplo realizamos un mix de actualizaciones para mostrar como se podria ejecutar.

coleccion.update(
    ids=["doc_1","doc_4"],
    documents=["Carrera de 18 km con 390 metros de altimetría positiva.","Esta es otra linea que modificamos y vemos como queda"], # Nuevo texto (recalcula el embedding)
    metadatas=[{"altimetria": 390, "distancia": "18"},None] # Nuevos metadatos
)

In [ ]:
# Vamos a usar el metodo GET para retornar un metadato especifico, en este caso el ids pero podria ser cualquier metadato como si se tratase de una busqueda SQL

coleccion.get(ids=["doc_4"])

{'ids': ['doc_2'],
 'embeddings': None,
 'documents': ['Las carreras intermedias suelen ser las mas tecnicas.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'categoria': 'Trail', 'altimetria': 3000, 'distancia': 60}]}

## ELIMINAR REGISTROS

In [ ]:
coleccion.delete(ids=["doc_1", "doc_2"])

{'deleted': 2}

In [ ]:
# ELIMINAMOS POR METADATOS

coleccion.delete(
    where={"categoria": "Calle"}
)

{'deleted': 1}

## MODO PERSISTENTE

Vamos a ver como se realiza la instanciación del cliente de ChromaDB asignando un path para que guarde los datos, lo que realizara es crear una base SQLITE con registros vectoriales.

In [ ]:
import chromadb

#Probamos modo persistente

client = chromadb.PersistentClient(path="./mi_base_de_datos")



In [ ]:
# Crea o accede a una colección
collection = client.create_collection(name="documentos")

# Guarda datos
collection.add(
    documents=[
        "El entrenamiento de pasadas en pista mejora la velocidad de crucero y el consumo máximo de oxígeno (VO2 máx).",
        "Para correr un ultramaratón de montaña es clave dominar el ritmo de caminata en subidas empinadas.",
        "La suplementación con creatina favorece la recuperación muscular y la fuerza explosiva en entrenamientos breves.",
        "El algoritmo HNSW organiza los vectores en capas jerárquicas para acelerar la búsqueda de vecinos cercanos.",
        "Las bases de datos vectoriales son la infraestructura central para los sistemas de recuperación mejorada por generación (RAG).",
        "Una carrera de 18 km con 390 metros de altimetría positiva es ideal para trabajar el ritmo continuo en terreno mixto.",
        "El filtrado por metadatos en ChromaDB reduce el espacio de búsqueda antes o después de calcular la distancia vectorial.",
        "La hidratación estratégica en carreras largas debe incluir sodio y electrolitos para evitar la hiponatremia.",
        "Para calcular el retorno de inversión (ROI) en campañas digitales se cruzan los datos de inversión con las conversiones de Google Analytics.",
        "Los Large Language Models (LLMs) utilizan vectores de alta dimensión para capturar el significado semántico de las palabras.",
        "El entrenamiento de fuerza en gimnasio previene lesiones comunes de rodilla y tobillo en corredores de trail.",
        "En Power BI, las funciones DAX permiten calcular medidas dinámicas acumuladas a lo largo de un período de tiempo.",
        "La carga de carbohidratos los días previos a una competencia de ultra distancia asegura los depósitos de glucógeno muscular.",
        "El método .upsert() en ChromaDB actualiza el documento si el ID existe o lo inserta si es un identificador nuevo.",
        "Las zapatillas de trail running necesitan un buen agarre o 'grip' para evitar resbalones en terrenos de piedra suelta o barro.",
        "Una correcta estrategia de contenido digital se basa en tres pilares: alcance orgánico, pauta digital y fidelización.",
        "La tasa de rebote en un sitio web puede indicar que el contenido no coincide con la intención de búsqueda del usuario.",
        "El sobreentrenamiento se detecta por una fatiga crónica, alteración del sueño y variaciones inusuales en la frecuencia cardíaca en reposo.",
        "Un script de Python que utiliza el EphemeralClient de ChromaDB almacena toda la información estrictamente en la memoria RAM.",
        "La preparación mental y la gestión del esfuerzo son tan importantes como el entrenamiento físico en carreras de 100 millas."
    ],
    metadatas=[
        {"categoria": "Entrenamiento", "fuente": "manual", "dificultad": "media"},
        {"categoria": "Trail", "fuente": "manual", "dificultad": "alta"},
        {"categoria": "Salud", "fuente": "web", "dificultad": "baja"},
        {"categoria": "Tecnología", "fuente": "web", "dificultad": "alta"},
        {"categoria": "Tecnología", "fuente": "manual", "dificultad": "alta"},
        {"categoria": "Trail", "fuente": "manual", "dificultad": "media"},
        {"categoria": "Tecnología", "fuente": "manual", "dificultad": "media"},
        {"categoria": "Salud", "fuente": "web", "dificultad": "media"},
        {"categoria": "Marketing", "fuente": "web", "dificultad": "media"},
        {"categoria": "Tecnología", "fuente": "web", "dificultad": "alta"},
        {"categoria": "Entrenamiento", "fuente": "manual", "dificultad": "baja"},
        {"categoria": "Marketing", "fuente": "manual", "dificultad": "alta"},
        {"categoria": "Salud", "fuente": "manual", "dificultad": "media"},
        {"categoria": "Tecnología", "fuente": "web", "dificultad": "baja"},
        {"categoria": "Trail", "fuente": "web", "dificultad": "baja"},
        {"categoria": "Marketing", "fuente": "manual", "dificultad": "media"},
        {"categoria": "Marketing", "fuente": "web", "dificultad": "baja"},
        {"categoria": "Salud", "fuente": "manual", "dificultad": "alta"},
        {"categoria": "Tecnología", "fuente": "manual", "dificultad": "baja"},
        {"categoria": "Trail", "fuente": "manual", "dificultad": "alta"}
    ],
    ids=[
        "doc_01", "doc_02", "doc_03", "doc_04", "doc_05",
        "doc_06", "doc_07", "doc_08", "doc_09", "doc_10",
        "doc_11", "doc_12", "doc_13", "doc_14", "doc_15",
        "doc_16", "doc_17", "doc_18", "doc_19", "doc_20"
    ]
)


In [ ]:
resultados = collection.query(
    query_texts=["documento"],
    n_results=2
)


for resultado in resultados["documents"][0]:
  print("-----")
  print(resultado)

-----
El método .upsert() en ChromaDB actualiza el documento si el ID existe o lo inserta si es un identificador nuevo.
-----
Una correcta estrategia de contenido digital se basa en tres pilares: alcance orgánico, pauta digital y fidelización.


In [ ]:
!ls mi_base_de_datos/

8ee4e52e-1385-4dec-ac54-08ffbb88cb30  f0079d23-b0fb-48f5-927d-83171e30eaa8
chroma.sqlite3
